In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *

# make sure table exists so we will need to import from delta.tables( This is now in utilities)
# from delta.tables import DeltaTable
# from pyspark.sql import DataFrame

In [0]:
%run "../utilities/Delta Merge"

In [0]:
bronze_source = spark.read.table('lakehouse.02_bronze.earthquake')
bronze_source.display()

# In the features there is an array of objects

In [0]:
# we are going to explode the array of objects to get multiple rows of those objects
# within each object we are only interested in the'properties' section and 'geometry' coordinates and the id
bronze_source_exploded = bronze_source.select(F.explode('features').alias('features'))
bronze_source_exploded.display()

In [0]:
# we are going to select the properties, geometry coordinates and id from the exploded features

bronze_source_exploded_v1 = bronze_source_exploded.select(
    "features.properties.*",
    "features.id",
    F.col("features.geometry.coordinates")[0].alias("longitude"),
    F.col("features.geometry.coordinates")[1].alias("latitude"),
    F.col("features.geometry.coordinates")[2].alias("depth")
)
bronze_source_exploded_v1.display()

In [0]:
# Clean up by fixing time columns from epoch to timestamp and casting some columns to double
source_transform = (
    bronze_source_exploded_v1.withColumn("time",F.from_unixtime(F.col("time")/1000).cast("timestamp"))
    .withColumn("updated",F.from_unixtime(F.col("updated")/1000).cast("timestamp"))
    .withColumn("nst", F.col("nst").cast("double"))
    .withColumn("sig", F.col("sig").cast("double"))
    .withColumn("tsunami", F.col("tsunami").cast("double"))
    .withColumn("felt", F.col("felt").cast("double"))
)
source_transform.display()

In [0]:
# Clean up some more by only selecting columns that we want

silver_source = (
    source_transform.select("id", "time", "longitude", "latitude", "depth", "mag", "magType", "place", "gap", "dmin", "rms", "net","code", "ids", "sources","types","nst","title","status","tsunami","sig","felt","updated")
).dropDuplicates()
silver_source.display()

## Testing

In [0]:
# Making sure we dont have any duplicates

# See how many duplicates we have
silver_source.groupBy("id").count().filter("count > 1").display()

# Now go back up to the code above and a drop duplicates line for all the columns we selected

# We may still see some duplicates as new files come in, because most rows will remains the same while one column changes suchas 'felt', you can look at the timestamp to get the latest one



In [0]:
# by spot checking some of these remaining duplicates, we can see that the 'felt' value is getting updated, so we will keep the latest one
silver_source.filter("id = 'nc75333407'").display()


#### To get the latest updates
- group by id
- order by update timestamp desc
- add row_number
- filter row_number = 1

In [0]:
from pyspark.sql.window import Window # Windows functions

window_split = Window.orderBy(F.col("updated").desc()).partitionBy("id")
silver_source_deduped_final = (silver_source
                    .withColumn("row_number", F.row_number().over(window_split))
                    .filter("row_number = 1")
                    .withColumn('hash_id', F.sha2(F.concat_ws('_', F.col('id'), F.col('time')),256)) #-> to guarantee uniqueness by generating a hash id based on two different columns
                    .drop("row_number")
                    )
silver_source_deduped_final.display()



In [0]:
# Now we define our silver layer table

silver_table_name = "lakehouse.03_silver.earthquake_events"

# function to check if table exists
# def check_delta_table(table_name):
#     return spark.catalog.tableExists(table_name) --> This function is now in the utilities folder
#check_delta_table(silver_table_name)

### Delta Merge with INSERT, UPDATE, or DELETE

In [0]:

# THIS MERGE FUNCTION HAS BEEN MOVED TO UTILITIES FOLDER TO BE CALLED UPON IN OTHER NOTEBOOKS

# def delta_merge(
#     source_table_name: DataFrame,
#     target_table_name: str,
#     primary_key: str,
# ):
    
#     target_dt = DeltaTable.forName(spark, target_table_name)
#     cols_map = {col: f"source.{col}" for col in source_table_name.columns} # updating all column names we gathered earlier with source.col_name, per example code below
#     merge_condition = f"target.{primary_key} = source.{primary_key}"

#     target_dt.alias("target").merge(
#         source_table_name.alias("source"),
#         merge_condition
#     ).whenNotMatchedInsert( # When primary key not matched then insert
#         values=cols_map
#     ).whenMatchedUpdate( # When primary key matched then update
#         set=cols_map
#     ).execute()
#     people_table.alias("target").merge(
#     new_df.alias("source"), "target.id = source.id"
# ).whenNotMatchedInsert(
#     condition='source._op = "INSERT"',
#     values={"id": "source.id", "name": "source.name", "age": "source.age"},
# ).whenMatchedUpdate(
#     condition='source._op = "UPDATE"',
#     set={"id": "source.id", "name": "source.name", "age": "source.age"},
# ).whenMatchedDelete(
#     condition='source._op = "DELETE"'
# ).execute()



In [0]:
# See if we need to create whole table if it doesn't exist, otherwise we will merge with existing table

if check_delta_table(silver_table_name):
    print(f"{silver_table_name} table exists in catalog, updating it")
    delta_merge(silver_source_deduped_final, silver_table_name, "hash_id") # using hash id created earlier
else:
    print(f"{silver_table_name} table does not exist in catalog, writing it for first time")
    silver_source_deduped_final.write.mode("overwrite").format("delta").option("delta.enableChangeDataFeed", "true").saveAsTable(silver_table_name)

In [0]:
%sql
-- select * from lakehouse.`03_silver`.earthquake_events
-- See if table is CDC enabled by looking at tableFeatures column, should have changeDataFeed enabled
desc detail lakehouse.03_silver.earthquake_events
